In [1]:
from pathlib import Path

import util

import automech

tag = util.notebook_prefix()

In [2]:
DATA_PATH = Path("../data")
CALC_PATH = Path(f"../{tag}/data/CKIN")

In [3]:
# Read mechanisms
par_mech0 = automech.io.read(DATA_PATH / "full_raw.json")
cal_mech0 = automech.io.read(DATA_PATH / f"{tag}.json")

In [4]:
# Add calculated thermo to mechanism object
*_, therm_file = CALC_PATH.glob("all_therm.ckin*")
cal_mech = automech.io.chemkin.update.thermo(cal_mech0, therm_file)

In [5]:
# Add calculated rates to mechanism object (use units of parent)
rate_files = list(CALC_PATH.glob("*.ckin"))
cal_mech = automech.set_rate_units(cal_mech, automech.rate_units(par_mech0))
cal_mech = automech.io.chemkin.update.rates(cal_mech, rate_files)

In [6]:
# Merge updated rates and thermo into parent mechanism
mech0 = automech.expand_parent_stereo(par_mech0, cal_mech)
mech = automech.update(mech0, cal_mech)

  0%|          | 0/2052 [00:00<?, ?it/s]

In [7]:
# Write
full0 = f"full_{tag}_control"
full = f"full_{tag}_calc"
part = f"{tag}_calc"
automech.io.write(mech0, DATA_PATH / f"{full0}.json")
automech.io.write(mech, DATA_PATH / f"{full}.json")
automech.io.write(cal_mech, DATA_PATH / f"{part}.json")
automech.io.chemkin.write.mechanism(mech0, DATA_PATH / "chemkin" / f"{full0}.dat")
automech.io.chemkin.write.mechanism(mech, DATA_PATH  / "chemkin" / f"{full}.dat")
automech.io.chemkin.write.mechanism(cal_mech, DATA_PATH  / "chemkin" / f"{part}.dat")

'ELEMENTS\n\nC\nH\nO\n\nEND\n\n\nSPECIES\n\nOH(4)         ! SMILES: [OH]                AMChI: AMChI=1/HO/h1H\nC5H8(522)     ! SMILES: C1=CCCC1            AMChI: AMChI=1/C5H8/c1-2-4-5-3-1/h1-2H,3-5H2\nC5H9O(852)r0  ! SMILES: O[C@H]1[CH]CCC1     AMChI: AMChI=1/C5H9O/c6-5-1-2-3-4-5/h1,5-6H,2-4H2/t5-/m0/s1\nC5H9O(852)r1  ! SMILES: O[C@@H]1[CH]CCC1    AMChI: AMChI=1/C5H9O/c6-5-1-2-3-4-5/h1,5-6H,2-4H2/t5-/m1/s1\nS(1248)r0     ! SMILES: O[C@H]1C[CH]CC1     AMChI: AMChI=1/C5H9O/c6-5-3-1-2-4-5/h1,5-6H,2-4H2/t5-/m0/s1\nS(1248)r1     ! SMILES: O[C@@H]1C[CH]CC1    AMChI: AMChI=1/C5H9O/c6-5-3-1-2-4-5/h1,5-6H,2-4H2/t5-/m1/s1\nS(1247)       ! SMILES: O[C]1CCCC1          AMChI: AMChI=1/C5H9O/c6-5-3-1-2-4-5/h6H,1-4H2\nCPTOJ(880)    ! SMILES: [O]C1CCCC1          AMChI: AMChI=1/C5H9O/c6-5-3-1-2-4-5/h5H,1-4H2\nC5H9O(878)r0  ! SMILES: C=C[C@@H](C[CH2])O  AMChI: AMChI=1/C5H9O/c1-3-5(6)4-2/h3,5-6H,1-2,4H2/t5-/m0/s1\nC5H9O(878)r1  ! SMILES: C=C[C@H](C[CH2])O   AMChI: AMChI=1/C5H9O/c1-3-5(6)4-2/h3,5-6H,1-2,4H

In [8]:
# Visualize intersections with the calculated mechanism
int_par_mech0 = automech.intersection(
    par_mech0, cal_mech, stereo=False
)
int_cal_mech = automech.intersection(
    par_mech0, cal_mech, right=True, stereo=False
)
dif_cal_mech = automech.difference(
    par_mech0, cal_mech, right=True, stereo=False
)
int_mech = automech.intersection(mech, cal_mech, stereo=False)
print("\n1. Original (compare):")
print(automech.io.chemkin.write.reactions_block(int_par_mech0, frame=False))
print("\n2. Calculated (compare):")
print(automech.io.chemkin.write.reactions_block(int_cal_mech, frame=False))
if not automech.reaction_count(dif_cal_mech) == 0:
    print("\n3. Calculated (new):")
    print(automech.io.chemkin.write.reactions_block(dif_cal_mech, frame=False))
    print("\n\n4. Calculated (all together):")
    print(automech.io.chemkin.write.reactions_block(int_mech, frame=False))
    automech.display(int_par_mech0)
    automech.display(int_mech)


1. Original (compare):
C5H9O(859) = CPTOJ(880)                   2.740E+09      0.000      6.900
C5H8(522) + OH(4) = C5H9O(852)            1.000E+00      0.000      0.000
    PLOG  /                      0.01000  3.650E+77     -20.00      33.87/
    PLOG  /                      0.01000  9.700E+59     -15.51      12.90/
    PLOG  /                       0.1000  1.380E+68     -17.01      27.91/
    PLOG  /                       0.1000  1.800E+56     -14.04      12.95/
    PLOG  /                        1.000  2.600E+59     -14.17      23.08/
    PLOG  /                        1.000  2.070E+50     -12.04      11.49/
    PLOG  /                        10.00  1.010E+54     -12.23      22.98/
    PLOG  /                        10.00  8.550E+41     -9.350      8.921/
    PLOG  /                        100.0  1.920E+48     -10.23      23.77/
    PLOG  /                        100.0  3.060E+32     -6.310      6.088/
C5H9O(853) = C5H9O(852)                   1.650E+07      1.020      14.20
C5H9